In [1]:
from weights_cuda import WeightMatrixCUDA as WeightMatrix
from topologies import square_torus
import cupy as np
import warnings
#warnings.filterwarnings("error")


In [2]:
def step_simulation(R, V, I, t=0, Delta=1, Eta=-5, J=15, tau=1, dt=1e-3):
    # Heun (RK2) instead of Euler + finite guards
    fR = Delta/np.pi + 2*R*V
    fV = V**2 + Eta + J*R + I - (np.pi**2)*(R**2)  # NOTE: +J*R (match your LaTeX)
    R1 = R + dt*fR
    V1 = V + dt*fV #np.clip(dt*fV, -1e8, 1e8)
    fR1 = Delta/np.pi + 2*R1*V1
    fV1 = V1**2 + Eta + J*R1 + I - (np.pi**2)*(R1**2)
    dR = 0.5*dt*(fR + fR1)
    #np.nan_to_num(dR, copy=False, nan=0.0, posinf=1e6, neginf=-1e6)
    #dR = np.clip(dR, -R/10, 1)
    R += dR*(10-R)/10
    #R[:] = np.clip(R, 1e-5, 10)
    dV = 0.5*dt*(fV + fV1)#*(100-np.abs(V))/100
    #np.nan_to_num(dV, copy=False, nan=0.0, posinf=1e6, neginf=-1e6)
    #dV = np.clip(dV, -1, 1)
    V += dV
    #V[:] = np.clip(V, -1e2, 1e2)

    #np.nan_to_num(R, copy=False, nan=0.0, posinf=1e6, neginf=-1e6)
    #np.nan_to_num(V, copy=False, nan=0.0, posinf=1e6, neginf=-1e6)
    return t + dt, dR

def f(R, V, I, Delta=1.0, Eta=-5.0, J=15.0):
    dR = Delta/np.pi + 2*R*V
    dV = V*V + Eta + J*R + I - (np.pi**2)*(R*R)
    return dR, dV

def step_simulation(R, V, I, t, dt, Delta=1.0, Eta=-5.0, J=15.0):
    k1R, k1V = f(R, V, I, Delta, Eta, J)
    k2R, k2V = f(R + 0.5*dt*k1R, V + 0.5*dt*k1V, I, Delta, Eta, J)
    k3R, k3V = f(R + 0.5*dt*k2R, V + 0.5*dt*k2V, I, Delta, Eta, J)
    k4R, k4V = f(R + dt*k3R,     V + dt*k3V,     I, Delta, Eta, J)

    R += (dt/6)*(k1R + 2*k2R + 2*k3R + k4R)
    V += (dt/6)*(k1V + 2*k2V + 2*k3V + k4V)
    return t + dt


rk4_step = np.ElementwiseKernel(
'float32 R, float32 V, float32 I, float32 dt, float32 Delta, float32 Eta, float32 J',
    'float32 R_out, float32 V_out',
    r'''
    const float pi  = 3.14159265358979323846f;
    const float pi2 = pi * pi;

    float k1R = Delta / pi + 2.0f * R * V;
    float k1V = V * V + Eta + J * R + I - pi2 * R * R;

    float R2 = R + 0.5f * dt * k1R;
    float V2 = V + 0.5f * dt * k1V;
    float k2R = Delta / pi + 2.0f * R2 * V2;
    float k2V = V2 * V2 + Eta + J * R2 + I - pi2 * R2 * R2;

    float R3 = R + 0.5f * dt * k2R;
    float V3 = V + 0.5f * dt * k2V;
    float k3R = Delta / pi + 2.0f * R3 * V3;
    float k3V = V3 * V3 + Eta + J * R3 + I - pi2 * R3 * R3;

    float R4 = R + dt * k3R;
    float V4 = V + dt * k3V;
    float k4R = Delta / pi + 2.0f * R4 * V4;
    float k4V = V4 * V4 + Eta + J * R4 + I - pi2 * R4 * R4;

    float fac = dt / 6.0f;
    R_out = R + fac * (k1R + 2.0f * k2R + 2.0f * k3R + k4R);
    V_out = V + fac * (k1V + 2.0f * k2V + 2.0f * k3V + k4V);
    ''',
    'rk4_step_mpr_f32'
)

def step_simulation(R, V, I, t, dt, Delta=1.0, Eta=-5.0, J=15.0):
    R[:], V[:] = rk4_step(R, V, I, dt, Delta, Eta, J)
    return t + dt


syn_kernel_src = r'''
extern "C" __global__
void syn_update(
    const float* __restrict__ R,
    const float* __restrict__ U,
    const float* __restrict__ V,
    const float* __restrict__ I_decayed,
    const int*   __restrict__ parents,
    const int*   __restrict__ children,
    int k,
    int E,
    float* __restrict__ I_out
){
    int e = blockDim.x * blockIdx.x + threadIdx.x;
    if (e >= E) return;

    int p = parents[e];
    int c = children[e];

    const float* u = U + (size_t)p * k;
    const float* v = V + (size_t)c * k;

    float dot = 0.0f;
    for (int kk = 0; kk < k; ++kk) {
        dot += u[kk] * v[kk];
    }

    float contrib = R[p] * dot - 0.5f * I_decayed[p];

    atomicAdd(&I_out[c], contrib);
}
''';

syn_update = np.RawKernel(syn_kernel_src, 'syn_update')




In [3]:
import threading, queue
import ipywidgets as w

Z = 50

t = 0
weights = WeightMatrix(square_torus(Z),
                       weight_initializer=lambda **s: (0.25*np.ones(s['size'])), rank=2)
                       
#children = np.array([*weights.network.values()])



R, V = np.zeros((2, weights.size), dtype=np.float32)
I = np.zeros(weights.size, dtype=np.float32)
weights.check = True


KeyboardInterrupt: 

In [56]:
square_torus(5)

{0: [1, 4, 5, 20],
 1: [2, 0, 6, 21],
 2: [3, 1, 7, 22],
 3: [4, 2, 8, 23],
 4: [0, 3, 9, 24],
 5: [6, 9, 10, 0],
 6: [7, 5, 11, 1],
 7: [8, 6, 12, 2],
 8: [9, 7, 13, 3],
 9: [5, 8, 14, 4],
 10: [11, 14, 15, 5],
 11: [12, 10, 16, 6],
 12: [13, 11, 17, 7],
 13: [14, 12, 18, 8],
 14: [10, 13, 19, 9],
 15: [16, 19, 20, 10],
 16: [17, 15, 21, 11],
 17: [18, 16, 22, 12],
 18: [19, 17, 23, 13],
 19: [15, 18, 24, 14],
 20: [21, 24, 0, 15],
 21: [22, 20, 1, 16],
 22: [23, 21, 2, 17],
 23: [24, 22, 3, 18],
 24: [20, 23, 4, 19]}

In [65]:
t=0
V.fill(0)
R.fill(0)
I.fill(0)

#children = np.array([*weights.network.values()])
#parents = np.arange(weights.size)[:, None]

def update_I(I, V, weights, t, children, parents, tau=8, dt=1e-3):
    #I.fill(0)

    I *= .2
    #I[0:Z] = 4 * np.sin(np.pi * t / 20)
    #I[-Z:0] = 4 * np.sin(np.pi * t / 20)
    #I *= 0.95
    delta = (R[:,None] * weights[parents, children] - I[:,None]/2)
    #delta *= dt
    np.add.at(I, children.ravel(), delta.ravel())
    I[0:Z] += 5 * np.sin(np.pi * t / 5)
    I[-Z:] += 5 * np.sin(np.pi * (t-1)/5)
    I[Z**2//2:Z**2//2+Z] = 10 * np.sin(np.pi * (t-1/2)/5)

decay = 0.2

def update_I(I, R, t, weights,
             parents_flat, children_flat,
             Z, decay=0.5):
    # decay
    I *= decay
    I_decayed = I.copy()  # snapshot for the kernel

    # zero recurrent part (keep decayed baseline)
    # we accumulate on top of I (already decayed)
    # if you want purely recurrent overwrite, use I_rec = cp.zeros_like(I)
    # and then I[...] = I_decayed + I_rec
    # here we just add to I in-place.

    U = weights.U
    Vw = weights.V

    E = parents_flat.size
    k = U.shape[1]

    threads = 256
    blocks = (E + threads - 1) // threads

    syn_update(
        (blocks,), (threads,),
        (R, U, Vw, I_decayed,
         parents_flat, children_flat,
         k, E,
         I)
    )

    # external drive (same as you had)
    #I[0:Z]        = 10 * np.sin(np.pi * t / 10)
    if t<50:
        I[-Z:]        = 10 * np.sin(np.pi * (t - 1) / 10)
    #I[Z**2//2:Z**2//2+Z] = 10 * np.sin(np.pi * (t - 0.5) / 10)


def update_W(weights, R, dR, A_p=5e-3, A_m=1e-3, tau_p=0.5, tau_m=.25):
    pass
    #i = np.arange(weights.size)[:, None]
    #j = np.array([*weights.network.values()])
    #weights.at[i, j] <<= A_p * R[:, None]*(R[j] + tau_p * dR[j]) - A_m * R[j]*(R[:,None] - tau_m*dR[:, None]) - 1e-2*np.clip(weights[i,j],-1e7,1e7)**3


import plotly.graph_objects as go, time

# initial setup
fig = go.FigureWidget()
fig.update_layout(width=500, height=500, margin=dict(l=0, r=0, b=0, t=0))

heat = fig.add_heatmap(z=np.zeros((Z,Z)).get(), colorscale="Viridis",zmin=0,zmax=8)

# make the figure a square
display(fig)
minvs = []
maxvs = []

# frame interval slider (interactive)
#frame_interval = w.IntSlider(value=100, min=1, max=1000, step=1, description="Frame N", continuous_update=True)

fig = go.FigureWidget()
fig.update_layout(width=500, height=500, margin=dict(l=0, r=0, b=0, t=0))

heat = fig.add_heatmap(
    z=np.zeros((Z, Z)).get(),
    colorscale="Viridis",
    zmin=0,
    zmax=5,
)

heat = fig.data[0]

display(fig)

frame_interval = w.IntSlider(
    value=250,
    min=1,
    max=1000,
    step=1,
    description="Frame N",
    continuous_update=False,  # <- change to False to reduce widget spam
)

display(frame_interval)

#children = np.array([*weights.network.values()])
#parents = np.arange(weights.size)[:, None]

# parents: shape (N, 1), children: shape (N, deg)
children = np.array([*weights.network.values()], dtype=np.int32)
parents  = np.arange(weights.size, dtype=np.int32)[:, None]

parents_flat  = parents.ravel()
children_flat = children.ravel()
from time import time as get_time

t1=get_time()
for tick in range(1000000):
    update_I(I, R, t, weights, parents_flat, children_flat, Z)
    t = step_simulation(R, V, I, t, dt=2e-3)

    N = max(1, int(frame_interval.value))

    if tick % N == 0:
        print(t)

        mP_grid = R.reshape(Z, Z)
        z_host = mP_grid.get()          # GPU -> CPU copy

        with fig.batch_update():
            heat.z = z_host
print(get_time()-t1)

FigureWidget({
    'data': [{'colorscale': [[0.0, '#440154'], [0.1111111111111111, '#482878'],
                             [0.2222222222222222, '#3e4989'], [0.3333333333333333,
                             '#31688e'], [0.4444444444444444, '#26828e'],
                             [0.5555555555555556, '#1f9e89'], [0.6666666666666666,
                             '#35b779'], [0.7777777777777778, '#6ece58'],
                             [0.8888888888888888, '#b5de2b'], [1.0, '#fde725']],
              'type': 'heatmap',
              'uid': '5c9d7293-8db4-4f95-9e80-b514e759596c',
              'z': {'bdata': ('AAAAAAAAAAAAAAAAAAAAAAAAAAAAAA' ... 'AAAAAAAAAAAAAAAAAAAAAAAAAAAAA='),
                    'dtype': 'f8',
                    'shape': '50, 50'},
              'zmax': 8,
              'zmin': 0}],
    'layout': {'height': 500, 'margin': {'b': 0, 'l': 0, 'r': 0, 't': 0}, 'template': '...', 'width': 500}
})

FigureWidget({
    'data': [{'colorscale': [[0.0, '#440154'], [0.1111111111111111, '#482878'],
                             [0.2222222222222222, '#3e4989'], [0.3333333333333333,
                             '#31688e'], [0.4444444444444444, '#26828e'],
                             [0.5555555555555556, '#1f9e89'], [0.6666666666666666,
                             '#35b779'], [0.7777777777777778, '#6ece58'],
                             [0.8888888888888888, '#b5de2b'], [1.0, '#fde725']],
              'type': 'heatmap',
              'uid': 'b79043e4-fb36-43e8-b168-8167dfa2e392',
              'z': {'bdata': ('AAAAAAAAAAAAAAAAAAAAAAAAAAAAAA' ... 'AAAAAAAAAAAAAAAAAAAAAAAAAAAAA='),
                    'dtype': 'f8',
                    'shape': '50, 50'},
              'zmax': 5,
              'zmin': 0}],
    'layout': {'height': 500, 'margin': {'b': 0, 'l': 0, 'r': 0, 't': 0}, 'template': '...', 'width': 500}
})

IntSlider(value=250, continuous_update=False, description='Frame N', max=1000, min=1)

0.002
0.5020000000000003
1.0020000000000007
1.5020000000000011
2.002000000000001
2.501999999999946
3.001999999999891
3.501999999999836
4.001999999999781
4.501999999999726
5.001999999999671
5.501999999999616
6.001999999999561
6.501999999999506
7.001999999999451
7.501999999999396
8.001999999999342
8.501999999999509
9.001999999999676
9.501999999999843
10.00200000000001
10.502000000000177
11.002000000000344
11.50200000000051
12.002000000000677
12.502000000000844
13.002000000001011
13.502000000001178
14.002000000001345
14.502000000001512
15.00200000000168
15.502000000001846
16.00200000000201
16.502000000001733
17.002000000001456
17.50200000000118
18.0020000000009
18.502000000000624
19.002000000000347
19.50200000000007
20.001999999999793
20.501999999999516
21.00199999999924
21.50199999999896
22.001999999998684
22.501999999998407
23.00199999999813
23.501999999997853
24.001999999997576
24.5019999999973
25.00199999999702
25.501999999996745
26.001999999996467
26.50199999999619
27.001999999995913

KeyboardInterrupt: 

ERROR: Could not find a version that satisfies the requirement free-threading (from versions: none)
ERROR: No matching distribution found for free-threading
Note: you may need to restart the kernel to use updated packages.


In [21]:
import cupy as cp
print(cp.cuda.runtime.getDeviceCount())


1


In [6]:
%conda install anywidget -y

Jupyter detected...
2 channel Terms of Service accepted
Channels:
 - conda-forge
 - defaults
Platform: linux-aarch64
Solving environment: done


==> WARNING: A newer version of conda exists. <==
    current version: 25.9.1
    latest version: 25.11.0

Please update conda by running

    $ conda update -n base -c defaults conda



## Package Plan ##

  environment location: /home/michael/miniconda3/envs/spike

  added / updated specs:
    - anywidget


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    anywidget-0.9.21           |     pyhd8ed1ab_0         105 KB  conda-forge
    psygnal-0.15.0             |     pyhd8ed1ab_0          73 KB  conda-forge
    wrapt-2.0.1                |  py314h51f160d_1          86 KB  conda-forge
    ------------------------------------------------------------
                                           Total:         263 KB

The following NEW packages will be 

In [ ]:
i=7
j=np.int64(list(weights.children[i]))
weights[[i],j]

In [ ]:
V

In [ ]:
weights[np.arange(10),np.arange(10,20)]

In [93]:
# set the IOPub message limit much higher
import os, json, sys

# Increase limits for the running IPython kernel process (best effort)
os.environ["IPYKERNEL_CELL_NAME"] = "high_iopub_limits"
try:
    from IPython import get_ipython

    ip = get_ipython()
    if ip is not None and hasattr(ip, "kernel") and hasattr(ip.kernel, "session"):
        # These config keys are read at startup; for a running kernel we adjust traitlets directly if present
        if hasattr(ip.kernel, "iopub_thread") and hasattr(ip.kernel.iopub_thread, "rate_limit"):
            # Disable rate limiting by setting huge limits
            ip.kernel.iopub_thread.rate_limit = 1e10
            ip.kernel.iopub_thread.max_msg_rate = 1e9
            ip.kernel.iopub_thread.max_msg_size = int(1e9)
except Exception as e:
    print("Could not adjust IOPub limits at runtime:", e, file=sys.stderr)

# Also tell Jupyter Server (if it respects env for spawned kernels later in this session)
os.environ["IPKernelApp.iopub_msg_rate_limit"] = "1000000000"
os.environ["IPKernelApp.iopub_data_rate_limit"] = "1.0e11"


In [90]:
ip.kernel.iopub_thread.rate_limit = 1e10

In [89]:
os.environ["IPKernelApp.iopub_msg_rate_limit"] = "1000000000"

In [ ]:
import numpy as np, time
import plotly.graph_objects as go

fig = go.FigureWidget([go.Scatter(x=[], y=[], mode="lines")])
display(fig)

y = []
for t in range(1000):
    y.append(np.sin(t/10))
    with fig.batch_update():
        fig.data[0].x = np.arange(len(y))
        fig.data[0].y = y
    time.sleep(0.001)


In [ ]:
 import ipywidgets as w, plotly.io as pio
print("ipywidgets", w.__version__)
_ = w.IntSlider()  # should render a slider
